## Code Attribution

Portions of this implementation build upon the **Flash** provenance-based intrusion detection system (PIDS).

> **Citation:**  
> Rehman, Muhammad Umer; Ahmadi, Hossein; and Hassan, Wajih Ul.  
> *A Context is Worth a Thousand Lies: Evading Intrusion Detectors via Intelligent Context Distortion.*  
> *In Proceedings of the 2024 IEEE Symposium on Security and Privacy (SP),* pp. 3571–3586. IEEE, 2024.  
> [https://github.com/DART-Laboratory/Flash-IDS](https://github.com/DART-Laboratory/Flash-IDS)

<details>
<summary>BibTeX Reference</summary>

```bibtex
@inproceedings{rehman2024contorter,
  title={A Context is Worth a Thousand Lies: Evading Intrusion Detectors via Intelligent Context Distortion},
  author={Rehman, Muhammad Umer and Ahmadi, Hossein and Hassan, Wajih Ul},
  booktitle={2024 IEEE Symposium on Security and Privacy (SP)},
  pages={3571--3586},
  year={2024},
  organization={IEEE}
}


In [ ]:
import pandas as pd
import torch
from torch_geometric.data import Data
import torch.nn.functional as F
import warnings
import json
import torch
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
import torch.nn as nn
from gensim.models import Word2Vec
from tqdm import tqdm
import torch.nn.functional as F
import math
import numpy as np
import sys
from collections import defaultdict
import gdown
import os
import re
import glob
import requests
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
%matplotlib inline

In [ ]:
urls = ["https://drive.google.com/file/d/1GG1aUnPjjzzdbxznVTN8X6oVfA-K4oIV/view?usp=drive_link"]
for url in urls:
    gdown.download(url, quiet=False, use_cookies=False, fuzzy=True)

### Code taken from the Flash Github repository to preprocess the data

In [ ]:
def extract_uuid(line):
    pattern_uuid = re.compile(r'uuid\":\"(.*?)\"')
    return pattern_uuid.findall(line)

def extract_subject_type(line):
    pattern_type = re.compile(r'type\":\"(.*?)\"')
    return pattern_type.findall(line)

def show(file_path):
    print(f"Processing {file_path}")

def extract_edge_info(line):
    pattern_src = re.compile(r'subject\":{\"com.bbn.tc.schema.avro.cdm18.UUID\":\"(.*?)\"}')
    pattern_dst1 = re.compile(r'predicateObject\":{\"com.bbn.tc.schema.avro.cdm18.UUID\":\"(.*?)\"}')
    pattern_dst2 = re.compile(r'predicateObject2\":{\"com.bbn.tc.schema.avro.cdm18.UUID\":\"(.*?)\"}')
    pattern_type = re.compile(r'type\":\"(.*?)\"')
    pattern_time = re.compile(r'timestampNanos\":(.*?),')

    edge_type = extract_subject_type(line)[0]
    timestamp = pattern_time.findall(line)[0]
    src_id = pattern_src.findall(line)

    if len(src_id) == 0:
        return None, None, None, None, None

    src_id = src_id[0]
    dst_id1 = pattern_dst1.findall(line)
    dst_id2 = pattern_dst2.findall(line)

    if len(dst_id1) > 0 and dst_id1[0] != 'null':
        dst_id1 = dst_id1[0]
    else:
        dst_id1 = None

    if len(dst_id2) > 0 and dst_id2[0] != 'null':
        dst_id2 = dst_id2[0]
    else:
        dst_id2 = None

    return src_id, edge_type, timestamp, dst_id1, dst_id2

def process_data(file_path):
    id_nodetype_map = {}
    notice_num = 1000000
    for i in range(100):
        now_path = file_path + '.' + str(i)
        if i == 0:
            now_path = file_path
        if not os.path.exists(now_path):
            break

        with open(now_path, 'r') as f:
            show(now_path)
            cnt = 0
            for line in f:
                cnt += 1
                if cnt % notice_num == 0:
                    print(cnt)

                if 'com.bbn.tc.schema.avro.cdm18.Event' in line or 'com.bbn.tc.schema.avro.cdm18.Host' in line:
                    continue

                if 'com.bbn.tc.schema.avro.cdm18.TimeMarker' in line or 'com.bbn.tc.schema.avro.cdm18.StartMarker' in line:
                    continue

                if 'com.bbn.tc.schema.avro.cdm18.UnitDependency' in line or 'com.bbn.tc.schema.avro.cdm18.EndMarker' in line:
                    continue

                uuid = extract_uuid(line)[0]
                subject_type = extract_subject_type(line)

                if len(subject_type) < 1:
                    if 'com.bbn.tc.schema.avro.cdm18.MemoryObject' in line:
                        id_nodetype_map[uuid] = 'MemoryObject'
                        continue
                    if 'com.bbn.tc.schema.avro.cdm18.NetFlowObject' in line:
                        id_nodetype_map[uuid] = 'NetFlowObject'
                        continue
                    if 'com.bbn.tc.schema.avro.cdm18.UnnamedPipeObject' in line:
                        id_nodetype_map[uuid] = 'UnnamedPipeObject'
                        continue

                id_nodetype_map[uuid] = subject_type[0]

    return id_nodetype_map

def process_edges(file_path, id_nodetype_map):
    notice_num = 1000000
    not_in_cnt = 0

    for i in range(100):
        now_path = file_path + '.' + str(i)
        if i == 0:
            now_path = file_path
        if not os.path.exists(now_path):
            break

        with open(now_path, 'r') as f, open(now_path+'.txt', 'w') as fw:
            cnt = 0
            for line in f:
                cnt += 1
                if cnt % notice_num == 0:
                    print(cnt)

                if 'com.bbn.tc.schema.avro.cdm18.Event' in line:
                    src_id, edge_type, timestamp, dst_id1, dst_id2 = extract_edge_info(line)

                    if src_id is None or src_id not in id_nodetype_map:
                        not_in_cnt += 1
                        continue

                    src_type = id_nodetype_map[src_id]

                    if dst_id1 is not None and dst_id1 in id_nodetype_map:
                        dst_type1 = id_nodetype_map[dst_id1]
                        this_edge1 = f"{src_id}\t{src_type}\t{dst_id1}\t{dst_type1}\t{edge_type}\t{timestamp}\n"
                        fw.write(this_edge1)

                    if dst_id2 is not None and dst_id2 in id_nodetype_map:
                        dst_type2 = id_nodetype_map[dst_id2]
                        this_edge2 = f"{src_id}\t{src_type}\t{dst_id2}\t{dst_type2}\t{edge_type}\t{timestamp}\n"
                        fw.write(this_edge2)

def run_data_processing():
    os.system('tar -zxvf ta1-trace-e3-official-1.json.tar.gz')

    path_list = ['ta1-trace-e3-official-1.json']

    for path in path_list:
        id_nodetype_map = process_data(path)

        process_edges(path, id_nodetype_map)

    os.system('cp ta1-trace-e3-official-1.json.4.txt trace_test.txt')

In [ ]:
run_data_processing()

In [ ]:
directory = "."

keep_file = "ta1-trace-e3-official-1.json.4"

pattern = os.path.join(directory, "ta1-trace-e3-official-1.json*")

for file_path in glob.glob(pattern):
    if os.path.basename(file_path) != keep_file:
        try:
            os.remove(file_path)
            print(f"Deleted: {file_path}")
        except Exception as e:
            print(f"Error deleting {file_path}: {e}")

print("Deleted unnecessary files")

In [ ]:
def add_node_properties(nodes, node_id, properties):
    if node_id not in nodes:
        nodes[node_id] = []
    nodes[node_id].extend(properties)

def update_edge_index(edges, edge_index, index):
    for src_id, dst_id in edges:
        src = index[src_id]
        dst = index[dst_id]
        edge_index[0].append(src)
        edge_index[1].append(dst)

def prepare_graph(df):
    nodes, labels, edges = {}, {}, []
    dummies = {"SUBJECT_PROCESS":0, "MemoryObject":1, "FILE_OBJECT_CHAR":2, "FILE_OBJECT_FILE":3,
               "FILE_OBJECT_DIR":4, "SUBJECT_UNIT":5, "UnnamedPipeObject":6, "FILE_OBJECT_UNIX_SOCKET":7, 
               "SRCSINK_UNKNOWN":8, "FILE_OBJECT_LINK":9, "NetFlowObject":10, "FILE_OBJECT_BLOCK":11}
    
    for _, row in df.iterrows():
        action = row["action"]
        properties = [row['exec'], action] + ([row['path']] if row['path'] else [])
        
        actor_id = row["actorID"]
        add_node_properties(nodes, actor_id, properties)
        labels[actor_id] = dummies[row['actor_type']]

        object_id = row["objectID"]
        add_node_properties(nodes, object_id, properties)
        labels[object_id] = dummies[row['object']]

        edges.append((actor_id, object_id))

    features, feat_labels, edge_index, index_map = [], [], [[], []], {}
    for node_id, props in nodes.items():
        features.append(props)
        feat_labels.append(labels[node_id])
        index_map[node_id] = len(features) - 1

    update_edge_index(edges, edge_index, index_map)

    return features, feat_labels, edge_index, list(index_map.keys())

### Downloading the Ground truth from the FLASH github repository

In [ ]:
url = "https://raw.githubusercontent.com/DART-Laboratory/Flash-IDS/refs/heads/main/data_files/trace.json"

response = requests.get(url)

if response.status_code == 200:
    Ground_Truth_mal = set(json.loads(response.content))
    print("JSON loaded into Ground_Truth_mal successfully!")
    print(f"Number of entries: {len(Ground_Truth_mal)}")
else:
    print(f"Failed to download: {response.status_code}")
    sys.exit(0)

### Downloading the trained models for evaluation from the Flash Github repository

In [ ]:
api_url = "https://api.github.com/repos/DART-Laboratory/Flash-IDS/contents/trained_weights/trace"

os.makedirs("trained_weights/trace", exist_ok=True)

response = requests.get(api_url)
if response.status_code == 200:
    files = response.json()
    for file in files:
        if file["name"].endswith(".pth") or file["name"].endswith(".model"):
            download_url = file["download_url"]
            save_path = os.path.join("trained_weights/trace", file["name"])

            print(f"Downloading {file['name']}...")
            r = requests.get(download_url)
            with open(save_path, "wb") as f:
                f.write(r.content)
            print(f"Saved: {save_path}")

    print("\nAll .pth and .model files downloaded successfully!")
else:
    print(f"Failed to access repo: {response.status_code}")


In [ ]:
class GCN(torch.nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.conv1 = SAGEConv(in_channel, 32, normalize=True)
        self.conv2 = SAGEConv(32, out_channel, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return F.softmax(x, dim=1)
    
model = GCN(30,11).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)


In [ ]:
class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]


def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(20)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(30)
w2vmodel = Word2Vec.load("trained_weights/trace/word2vec_trace_E3.model")


In [ ]:
def add_attributes(d,p):
    
    f = open(p)
    data = [json.loads(x) for x in f if "EVENT" in x]

    info = []
    for x in data:
        try:
            action = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['type']
        except:
            action = ''
        try:
            actor = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['subject']['com.bbn.tc.schema.avro.cdm18.UUID']
        except:
            actor = ''
        try:
            obj = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject']['com.bbn.tc.schema.avro.cdm18.UUID']
        except:
            obj = ''
        try:
            timestamp = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['timestampNanos']
        except:
            timestamp = ''
        try:
            cmd = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['properties']['map']['exec']
        except:
            cmd = ''
        try:
            path = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObjectPath']['string']
        except:
            path = ''
        try:
            path2 = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject2Path']['string']
        except:
            path2 = ''
        try:
            obj2 = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject2']['com.bbn.tc.schema.avro.cdm18.UUID']
            info.append({'actorID':actor,'objectID':obj2,'action':action,'timestamp':timestamp,'exec':cmd, 'path':path2})
        except:
            pass

        info.append({'actorID':actor,'objectID':obj,'action':action,'timestamp':timestamp,'exec':cmd, 'path':path})

    rdf = pd.DataFrame.from_records(info).astype(str)
    d = d.astype(str)

    return d.merge(rdf,how='inner',on=['actorID','objectID','action','timestamp']).drop_duplicates()

In [ ]:
def Get_Adjacent(ids, mapp, edges, hops):
    if hops == 0:
        return set()
    neighbors = set()
    for edge in zip(edges[0], edges[1]):
        if any(mapp[node] in ids for node in edge): 
            neighbors.update(mapp[node] for node in edge)
    if hops > 1:
        neighbors = neighbors.union(Get_Adjacent(neighbors, mapp, edges, hops - 1))
    return neighbors

def calculate_metrics(TP, FP, FN, TN):
    FPR = FP / (FP + TN) if FP + TN > 0 else 0
    TPR = TP / (TP + FN) if TP + FN > 0 else 0
    TNR = TN / (TN + FP) if TN + FP > 0 else 0
    prec = TP / (TP + FP) if TP + FP > 0 else 0
    rec = TP / (TP + FN) if TP + FN > 0 else 0
    fscore = (2 * prec * rec) / (prec + rec) if prec + rec > 0 else 0
    return prec, rec, fscore, FPR, TPR, TNR

def helper(MP, all_pids, GP, edges, mapp):
    TP = MP.intersection(GP)  
    FP = MP - GP              
    FN = GP - MP              
    TN = all_pids - (GP | MP) 
    two_hop_gp = Get_Adjacent(GP, mapp, edges, 1)   
    two_hop_tp = Get_Adjacent(TP, mapp, edges, 1)     
    FPL = FP - two_hop_gp                            
    TPL = TP.union(FN.intersection(two_hop_tp))        
    FN = FN - two_hop_tp                      
    TP, FP, FN, TN = len(TPL), len(FPL), len(FN), len(TN)
    prec, rec, fscore, FPR, TPR, TNR = calculate_metrics(TP, FP, FN, TN)
    
    print(f"True Positives: {TP}, True Negatives: {TN}, False Positives: {FP}, False Negatives: {FN}")
    print(f"Precision: {round(prec, 2)}, Recall: {round(rec, 2)}, Fscore: {round(fscore, 2)}")
    print(f"False Positive Rate: {round(FPR, 2)}, True Positive Rate: {round(TPR, 2)}, True Negative Rate: {round(TNR, 2)}")

    return TP, TN, FP, FN, prec, rec, fscore


In [ ]:
f = open("trace_test.txt")
data = f.read().split('\n')
data = [line.split('\t') for line in data]
df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
df = df.dropna()
df.sort_values(by='timestamp', ascending=True,inplace=True)

df = add_attributes(df,"ta1-trace-e3-official-1.json.4")

phrases,labels,edges,mapp = prepare_graph(df)
nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)  

mapp_dict = {node_id: idx for idx, node_id in enumerate(mapp)}

indices_of_malicious_nodes = [mapp_dict[mid] for mid in Ground_Truth_mal if mid in mapp_dict]

all_ids = list(df['actorID']) + list(df['objectID'])
all_ids = set(all_ids)

In [ ]:
graph = Data(
    x=torch.tensor(nodes, dtype=torch.float).to(device),
    y=torch.tensor(labels, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges, dtype=torch.long).to(device)
)

graph.n_id = torch.arange(graph.num_nodes, device=device)
flag = torch.ones(graph.num_nodes, dtype=torch.bool, device=device)

for m_n in range(10):
    model.load_state_dict(torch.load(f'trained_weights/trace/lword2vec_gnn_trace{m_n}_E3.pth'))
    model.to(device)
    model.eval()

    graph = graph.to(device)
    out = model(graph.x, graph.edge_index)

    sorted, indices = out.sort(dim=1, descending=True)
    conf = (sorted[:, 0] - sorted[:, 1]) / sorted[:, 0]
    conf = (conf - conf.min()) / conf.max()

    pred = indices[:, 0]
    cond = (pred == graph.y)  & (conf >= 0.3)
    flag[graph.n_id[cond]] &= False

indices_of_flagged_nodes = torch.nonzero(flag, as_tuple=False).view(-1).tolist()

print("Number of Flagged Nodes:", len(indices_of_flagged_nodes))

ids = set([mapp[x] for x in indices_of_flagged_nodes])
alerts = helper(set(ids), set(all_ids), Ground_Truth_mal, edges, mapp)


# Step 1: Benign Node Selection
### Identify candidate nodes with the same label as potentially malicious nodes.

In [ ]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    r"""Groups malicious nodes based on their labels.

    :param malicious_indices: List of indices for malicious nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of malicious node indices as values.
    """

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)

malicious_groups = split_malicious_nodes_by_label(indices_of_malicious_nodes, labels)

print("The number of malicious nodes for each label is as follows:")

for label, nodes_indices in malicious_groups.items():
    print(f"Label {label}: {len(nodes_indices)}")


In [ ]:
def split_nodes_by_label_exclude_malicious(phrases, labels, indices_of_malicious_nodes):

    r"""Groups benign (non-malicious) nodes based on their labels.

    :param phrases: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :param indices_of_malicious_nodes: List of indices for malicious nodes.
    :return: Dictionary with label as key and list of benign node indices as values.
    """

    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)

    for node_idx, node_features in enumerate(phrases):
        if node_idx not in malicious_set:
            node_label = labels[node_idx]
            node_groups[node_label].append(node_idx)

    return dict(node_groups)

all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(phrases, labels, indices_of_malicious_nodes)

print("The number of benign nodes for each label is as follows:")

for label, nodess in all_nodes_grouped_excluding_malicious.items():
    print(f"Label {label}: {len(nodess)}")  


# Step 2: Minimal Interaction Selection
### Prioritize nodes with the fewest properties to reduce detectability.

In [ ]:
def filter_nodes_by_num_interactions(node_groups, phrases, min_len=0, max_len=5):

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        filtered_groups[label] = [
            idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len
        ]

    return filtered_groups

filtered_nodes_by_interactions = filter_nodes_by_num_interactions(all_nodes_grouped_excluding_malicious, phrases)

print("The number of candidate benign nodes for each label is as follows:")

for label, nodes_indices in filtered_nodes_by_interactions.items():
    print(f"Label {label}: {len(nodes_indices)}")


# Step 3: Contextual Consistency Filtering
### Choose nodes with higher cosine similarity to malicious nodes to preserve the contextual  integrity and stealth.

In [ ]:
def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):

    r"""Orders the benign nodes within each label based on their contextual similarity to the malicious nodes in the same label in descending order.

    :param malicious_groups: Dictionary with label as key and list of malicious node indices as values.
    :param filtered_nodes_grouped: Dictionary with label as key and list of filtered benign node indices as values.
    :param nodes: Feature vectors (embeddings) for all nodes.
    :return: Dictionary with label as key and list of sorted benign node indices as values.
    """

    top_nodes_by_similarity = {}

    for label in tqdm(malicious_groups.keys()):
        if label in filtered_nodes_grouped and len(filtered_nodes_grouped[label]) > 0:
            
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            similarities = cosine_similarity(filtered_vectors, malicious_vectors) 

            max_similarities = similarities.max(axis=1)  

            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  

            top_nodes_by_similarity[label] = sorted_indices

    return top_nodes_by_similarity

top_nodes_by_similarity = compute_top_similar_nodes(malicious_groups, filtered_nodes_by_interactions, nodes)

print("The number of candidate benign nodes for each label is as follows:")

for label, top_nodes in top_nodes_by_similarity.items():
    print(f"Label {label}: {len(top_nodes)}")  


# Step 4: Confidence-based Filtering
### Select benign nodes predicted with the highest confidence by the model.


In [ ]:
def confidence_filtering(top_nodes_by_similarity, nodes, labels, edges, model, device):

    r"""Determines the best benign node replacement for each label based on the maximum number of correct predictions across an ensemble of 22 models.

    :param top_nodes_by_similarity: Dictionary with label as key and list of sorted benign node indices as values.
    :param nodes: embeddings for all nodes.
    :param labels: List of labels for all nodes.
    :param edges: Edge index for the graph.
    :param model: GCN model.
    :param device: Device to run the evaluation on.
    :return: Dictionary with label as key and the most confident benign node index as value.
    """

    best_replacements = {}

    for target_label in list(top_nodes_by_similarity.keys()):
        previous_matches = 0
        max_matches_index = -1
        graph = Data(
            x=torch.tensor(nodes, dtype=torch.float).to(device),
            y=torch.tensor(labels, dtype=torch.long).to(device),
            edge_index=torch.tensor(edges, dtype=torch.long).to(device)
        )

        graph.n_id = torch.arange(graph.num_nodes, device=device)
        if len(top_nodes_by_similarity[target_label]) > 2000:
            top_nodes_by_similarity[target_label] = top_nodes_by_similarity[target_label][:2000]

        for current_index in tqdm(top_nodes_by_similarity[target_label]):
            target_index = current_index

            predictions = torch.zeros(10, dtype=torch.long, device=device)
            confidences = torch.zeros(10, dtype=torch.float, device=device)

            for m_n in range(10):
                model.load_state_dict(torch.load(f'trained_weights/trace/lword2vec_gnn_trace{m_n}_E3.pth'))
                model.to(device)
                model.eval()
                out = model(graph.x, graph.edge_index)

                sorted_out, indices = out.sort(dim=1, descending=True)
                conf = (sorted_out[:, 0] - sorted_out[:, 1]) / sorted_out[:, 0]
                conf = (conf - conf.min()) / (conf.max() + 1e-8)

                predictions[m_n] = indices[target_index, 0]
                confidences[m_n] = conf[target_index]

            true_label = graph.y[target_index]
            matches = ((predictions == true_label) ).sum().item()
            if matches > previous_matches:
                previous_matches = matches
                max_matches_index = current_index
                if matches == 10:  
                    break 

        best_replacements[target_label] = max_matches_index

        print(f"max matches index for label {target_label}:", max_matches_index)
        print("matches:", previous_matches)

    return best_replacements

best_benign_replacements = confidence_filtering(top_nodes_by_similarity, nodes, labels, edges, model, device)


# Graph Structure Adjustment

In [ ]:
def graph_modification(df, indices_of_malicious_nodes, best_benign_replacements, mapp, labels):

    r"""
    Modifies the graph by adding edges between malicious and selected benign nodes.
    :param df: Original DataFrame containing graph edges.
    :param indices_of_malicious_nodes: List of indices for malicious nodes.
    :param best_benign_replacements: Dictionary with label as key and the most confident benign node index as value.
    :param mapp: List mapping node indices to node IDs.
    :param labels: List of labels for all nodes.
    :return: Modified DataFrame with new edges added.
    """

    print("Number of rows in df before modification:", len(df))

    malicious_to_benign = {}

    for i in indices_of_malicious_nodes:
        label = labels[i]
        if label in best_benign_replacements and best_benign_replacements[label] not in [None,-1]:
            benign_index = best_benign_replacements[label]
            malicious_to_benign[mapp[i]] = mapp[benign_index]

    all_benign_replacements = set(malicious_to_benign.values())

    benign_rows_dict = {benign_id: df[(df['actorID'] == benign_id) | (df['objectID'] == benign_id)].copy()
                        for benign_id in all_benign_replacements}

    new_rows = []

    for malicious_id, benign_id in tqdm(malicious_to_benign.items(), desc="Processing malicious nodes"):
        if benign_id in benign_rows_dict:
            modified_rows = benign_rows_dict[benign_id].copy()  
            modified_rows.loc[modified_rows['actorID'] == benign_id, 'actorID'] = malicious_id
            modified_rows.loc[modified_rows['objectID'] == benign_id, 'objectID'] = malicious_id
            new_rows.append(modified_rows)

    df_curated = pd.concat([df] + new_rows, ignore_index=True)

    print("Number of rows in df after modification:", len(df_curated))

    print("The number of triggered events are: ", len(df_curated) - len(df))

    return df_curated

df_curated = graph_modification(df, indices_of_malicious_nodes, best_benign_replacements, mapp, labels)



In [ ]:
phrases, labels, edges, mapp = prepare_graph(df_curated)
nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)
all_ids = list(df_curated['actorID']) + list(df_curated['objectID'])
all_ids = set(all_ids)

In [ ]:
graph = Data(
    x=torch.tensor(nodes, dtype=torch.float).to(device),
    y=torch.tensor(labels, dtype=torch.long).to(device),
    edge_index=torch.tensor(edges, dtype=torch.long).to(device)
)

graph.n_id = torch.arange(graph.num_nodes, device=device)
flag = torch.ones(graph.num_nodes, dtype=torch.bool, device=device)

for m_n in range(10):
    model.load_state_dict(torch.load(f'trained_weights/trace/lword2vec_gnn_trace{m_n}_E3.pth'))
    model.to(device)
    model.eval()

    graph = graph.to(device)
    out = model(graph.x, graph.edge_index)

    sorted, indices = out.sort(dim=1, descending=True)
    conf = (sorted[:, 0] - sorted[:, 1]) / sorted[:, 0]
    conf = (conf - conf.min()) / conf.max()

    pred = indices[:, 0]
    cond = (pred == graph.y)
    flag[graph.n_id[cond]] &= False

indices_of_flagged_nodes = torch.nonzero(flag, as_tuple=False).view(-1).tolist()

print("Number of Flagged Nodes:", len(indices_of_flagged_nodes))

ids = set([mapp[x] for x in indices_of_flagged_nodes])
alerts = helper(set(ids), set(all_ids), Ground_Truth_mal, edges, mapp)

# OccVer
#### If the length of unqiue triplets (source, action, destination) in both original dataframe and curated dataframe are the same, then it means all added triplets (triggered interactions) occured before evasion.

In [ ]:
unique_exec_action_path_df = df[["exec", "action", "path"]].drop_duplicates()
print(len(unique_exec_action_path_df))

unique_exec_action_path_df = df_curated[["exec", "action", "path"]].drop_duplicates()
print(len(unique_exec_action_path_df))